In [ ]:
# =============================================================================
# ForestGuard — Export GEE Colab v2.2
#
# Sintoma reportado: só panel_*_2020.csv tem linhas; 2000–2019 só header.
#
# Causa provável: stack com bandas GHSL 100% mascaradas (year < 2020) fazendo
# sampleRegions não gerar features em alguns fluxos EE; ou Tasks antigas (v2)
# ainda no Drive misturadas com um 2020 bom.
#
# Correções v2.2:
# 1) Nunca inclui bandas totalmente mascaradas no Image amostrado.
#    built_* só entra no stack se year_t0 >= 2020; senão propriedades null no map.
# 2) Diagnóstico size() em TODOS os anos antes do Export (falha cedo se N==0).
# 3) Sem filtro forest_t0 no servidor.
# 4) Rótulo duro floresta→não-floresta; sem road_dist_km fantasma.
# =============================================================================

!pip -q install earthengine-api geobr geopandas pandas

from __future__ import annotations

import json
from pathlib import Path

import ee
import geobr
import pandas as pd

PROJECT_ID = "gen-lang-client-0232888542"
MUNICIPIOS = [4210100]
START_YEAR = 2000
END_YEAR = 2020
CELL_M = 1000
CRS = "EPSG:32722"
FOREST_CLASSES = [3, 4, 5, 6, 49]
FOREST_THRESHOLD = 0.50
LOSS_MIN_DROP = 0.0
MAX_STABILITY_YEARS = 6
NEIGHBOR_RADIUS_M = 1500
MIN_VALID_FRACTION = 0.80
GHSL_VALID_FROM_YEAR = 2020

EXPORT_FOLDER = "ForestGuard_Panels_v22"  # pasta NOVA — não misture com exports vazios
LOCAL_DIR = Path("/content/ForestGuard")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR = Path("/content/drive/MyDrive") / EXPORT_FOLDER

# Se True, size() em cada ano (mais lento, recomendado na 1ª rodada)
DIAGNOSE_EVERY_YEAR = True

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

MB_ASSET = (
    "projects/mapbiomas-public/assets/brazil/lulc/"
    "collection9/mapbiomas_collection90_integration_v1"
)
GHSL_ASSET = "JRC/GHSL/P2023A/GHS_BUILT_S/2020"
mb = ee.Image(MB_ASSET)


def assert_mapbiomas_bands(years):
    requested = [f"classification_{int(y)}" for y in sorted(set(years))]
    available = set(mb.bandNames().getInfo())
    missing = [b for b in requested if b not in available]
    if missing:
        raise ValueError("Bandas MapBiomas ausentes: " + ", ".join(missing))
    print(f"MapBiomas OK: {requested[0]} … {requested[-1]}")


def get_municipality_geom(ibge_code):
    gdf = geobr.read_municipality(code_muni=int(ibge_code), year=2022)
    if gdf.empty:
        raise ValueError(f"Município {ibge_code} não encontrado")
    return ee.Geometry(json.loads(gdf.to_json())["features"][0]["geometry"])


def forest_fraction_image(year):
    image = mb.select(f"classification_{int(year)}")
    valid = image.mask().gt(0)
    forest = image.remap(FOREST_CLASSES, [1] * len(FOREST_CLASSES), 0).unmask(0)
    forest_fraction = (
        forest.updateMask(valid)
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65536)
        .reproject(crs=CRS, scale=CELL_M)
        .rename(f"forest_frac_{int(year)}")
    )
    valid_fraction = (
        valid.unmask(0)
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65536)
        .reproject(crs=CRS, scale=CELL_M)
        .rename(f"valid_frac_{int(year)}")
    )
    return forest_fraction.addBands(valid_fraction)


def static_features():
    elev = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elev_m")
    slope = ee.Terrain.slope(elev).rename("slope_deg")
    return elev.addBands(slope)


def built_image_valid():
    """GHSL 2020 — só usar quando year_t0 >= GHSL_VALID_FROM_YEAR."""
    built = ee.Image(GHSL_ASSET).select("built_surface").rename("built_surface")
    urban = built.gt(50).unmask(0)
    dist_km = (
        urban.fastDistanceTransform(1024, "pixels", "squared_euclidean")
        .sqrt()
        .multiply(ee.Image.pixelArea().sqrt())
        .divide(1000)
        .rename("built_dist_km")
    )
    return built.addBands(dist_km)


def stable_forest_count(year):
    count = ee.Image.constant(0).rename("years_stable_forest")
    still_stable = ee.Image.constant(1)
    for back in range(MAX_STABILITY_YEARS):
        frac = forest_fraction_image(year - back).select(0).unmask(0)
        still_stable = still_stable.And(frac.gte(FOREST_THRESHOLD))
        count = count.add(still_stable)
    return count.rename("years_stable_forest")


def make_grid(geom, ibge_code):
    samples = ee.Image.constant(1).rename("grid").sample(
        region=geom,
        projection=ee.Projection(CRS).atScale(CELL_M),
        geometries=True,
        dropNulls=True,
        tileScale=4,
    )

    def add_id(f):
        xy = f.geometry().transform(CRS, 1).coordinates()
        x = ee.Number(xy.get(0)).divide(CELL_M).floor().multiply(CELL_M).toInt()
        y = ee.Number(xy.get(1)).divide(CELL_M).floor().multiply(CELL_M).toInt()
        center = ee.Geometry.Point(
            [x.add(CELL_M / 2), y.add(CELL_M / 2)], CRS
        ).transform("EPSG:4326", 1)
        ll = center.coordinates()
        return f.set(
            {
                "cell_id": ee.String("M")
                .cat(str(int(ibge_code)))
                .cat("_")
                .cat(x.format())
                .cat("_")
                .cat(y.format()),
                "ibge_code": str(int(ibge_code)),
                "lon": ll.get(0),
                "lat": ll.get(1),
                "grid_x_m": x,
                "grid_y_m": y,
            }
        ).setGeometry(center)

    return samples.map(add_id)


def hard_loss_band(frac0, frac_h, valid0, valid_h, name):
    ok = valid0.unmask(0).gte(MIN_VALID_FRACTION).And(
        valid_h.unmask(0).gte(MIN_VALID_FRACTION)
    )
    was_forest = frac0.unmask(0).gte(FOREST_THRESHOLD)
    not_forest_h = frac_h.unmask(0).lt(FOREST_THRESHOLD)
    transition = was_forest.And(not_forest_h)
    if LOSS_MIN_DROP and float(LOSS_MIN_DROP) > 0:
        transition = transition.And(
            frac0.unmask(0).subtract(frac_h.unmask(0)).gte(LOSS_MIN_DROP)
        )
    # 1/0 onde válido; -1 onde inválido (depois vira null)
    return ee.Image(transition).unmask(0).where(ok.Not(), -1).rename(name)


def build_year_collection(ibge_code, year, grid):
    year = int(year)
    f0 = forest_fraction_image(year)
    f1 = forest_fraction_image(year + 1)
    f2 = forest_fraction_image(year + 2)
    f3 = forest_fraction_image(year + 3)

    frac0 = f0.select(0).unmask(0)
    valid0 = f0.select(1).unmask(0)

    frac1, valid1 = f1.select(0), f1.select(1)
    frac2, valid2 = f2.select(0), f2.select(1)
    frac3, valid3 = f3.select(0), f3.select(1)

    forest_t0 = frac0.gte(FOREST_THRESHOLD).rename("forest_t0")

    loss1 = hard_loss_band(
        frac0, frac1, valid0, valid1, "y_loss_1y"
    )

    # Exige dados válidos no início e em todos os três anos futuros.
    valid3y = (
        valid0.gte(MIN_VALID_FRACTION)
        .And(valid1.unmask(0).gte(MIN_VALID_FRACTION))
        .And(valid2.unmask(0).gte(MIN_VALID_FRACTION))
        .And(valid3.unmask(0).gte(MIN_VALID_FRACTION))
    )

    # O alvo só é definido para células florestais no ano inicial.
    was_forest = frac0.gte(FOREST_THRESHOLD)

    # Perda em qualquer um dos três anos futuros.
    loss_any = (
        frac1.unmask(0).lt(FOREST_THRESHOLD)
        .Or(frac2.unmask(0).lt(FOREST_THRESHOLD))
        .Or(frac3.unmask(0).lt(FOREST_THRESHOLD))
    )

    loss3 = (
        ee.Image(loss_any)
        .unmask(0)
        .where(was_forest.Not(), -1)
        .where(valid3y.Not(), -1)
        .rename("y_loss_3y")
    )

    neighbor = frac0.reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=ee.Kernel.circle(radius=NEIGHBOR_RADIUS_M, units="meters"),
        skipMasked=False,
    ).unmask(0).rename("forest_neighbor_frac")

    # Stack CORE: só bandas sempre válidas no ponto da grade (nunca mask total)
    stack = (
        frac0.rename("forest_fraction_t0")
        .addBands(valid0.rename("valid_fraction_t0"))
        .addBands(forest_t0)
        .addBands(loss1)
        .addBands(loss3)
        .addBands(neighbor)
        .addBands(static_features())
        .addBands(stable_forest_count(year).unmask(0))
    )

    # GHSL: só adiciona bandas reais a partir de 2020 — evita Image 100% mask no stack
    use_built = year >= GHSL_VALID_FROM_YEAR
    if use_built:
        stack = stack.addBands(built_image_valid())

    sampled = stack.sampleRegions(
        collection=grid,
        properties=["cell_id", "ibge_code", "lon", "lat", "grid_x_m", "grid_y_m"],
        scale=CELL_M,
        projection=ee.Projection(CRS).atScale(CELL_M),
        geometries=False,
        tileScale=8,
    )

    def finalize(f):
        def fix_loss(feature, prop):
            v = ee.Number(feature.get(prop))
            return feature.set(prop, ee.Algorithms.If(v.eq(-1), None, v))

        f = f.set({"year_t0": year, "ibge_code": str(int(ibge_code))})
        f = fix_loss(f, "y_loss_1y")
        f = fix_loss(f, "y_loss_3y")
        # Anos < 2020: propriedades built ausentes (null), sem banda fantasma
        if not use_built:
            f = f.set({"built_surface": None, "built_dist_km": None})
        return f

    return sampled.map(finalize)


def diagnose(fc, label):
    n = int(fc.size().getInfo())
    print(f"  [diagnóstico] {label}: {n} features")
    if n == 0:
        raise RuntimeError(
            f"Collection VAZIA ({label}). Não exporte. "
            "Verifique CRS, município e se a grade tem células."
        )
    return n


# ---- main ----
needed = range(START_YEAR - (MAX_STABILITY_YEARS - 1), END_YEAR + 4)
assert_mapbiomas_bands(needed)

export_manifest = []
for ibge_code in MUNICIPIOS:
    print(f"\n=== Município {ibge_code} ===")
    geom = get_municipality_geom(ibge_code)
    grid = make_grid(geom, ibge_code)
    grid_n = int(grid.size().getInfo())
    print(f"Grade: {grid_n} células → pasta Drive: {EXPORT_FOLDER}")
    if grid_n == 0:
        raise RuntimeError("Grade vazia")

    for year in range(START_YEAR, END_YEAR + 1):
        fc = build_year_collection(ibge_code, year, grid)
        if DIAGNOSE_EVERY_YEAR:
            diagnose(fc, f"{ibge_code}/{year}")
        elif year in (START_YEAR, 2010, END_YEAR):
            diagnose(fc, f"{ibge_code}/{year}")

        desc = f"FG_v22_{ibge_code}_{year}"
        selectors = [
            "cell_id", "ibge_code", "lon", "lat", "year_t0",
            "forest_t0", "y_loss_1y", "y_loss_3y",
            "slope_deg", "elev_m", "forest_neighbor_frac",
            "years_stable_forest", "forest_fraction_t0", "valid_fraction_t0",
            "grid_x_m", "grid_y_m",
            "built_surface", "built_dist_km",
        ]
        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=desc,
            folder=EXPORT_FOLDER,
            fileNamePrefix=f"panel_{ibge_code}_{year}",
            fileFormat="CSV",
            selectors=selectors,
        )
        task.start()
        export_manifest.append(
            {"ibge_code": ibge_code, "year": year, "task_id": task.id, "description": desc}
        )
        print(f"  Export iniciado: {desc}")

(LOCAL_DIR / "export_manifest_v22.json").write_text(
    json.dumps(export_manifest, indent=2), encoding="utf-8"
)
print(f"\nManifesto salvo. Pasta Drive: {EXPORT_FOLDER}")
print("Apague CSVs vazios antigos. Só use arquivos desta pasta nova.")


def validate_downloaded_exports():
    reports = []
    for ibge_code in MUNICIPIOS:
        for year in range(START_YEAR, END_YEAR + 1):
            path = DRIVE_DIR / f"panel_{ibge_code}_{year}.csv"
            if not path.exists():
                print(f"FALTANDO: {path.name}")
                continue
            df = pd.read_csv(path)
            if len(df) == 0:
                print(f"VAZIO: {path.name}")
                continue
            reports.append(
                {
                    "file": path.name,
                    "rows": len(df),
                    "forest": int((df["forest_t0"] >= 0.5).sum()),
                    "pos_3y": int((df["y_loss_3y"] == 1).sum()),
                    "built_na_pct": round(100 * df["built_surface"].isna().mean(), 1)
                    if "built_surface" in df.columns
                    else None,
                }
            )
    if not reports:
        raise RuntimeError("Nenhum CSV com dados")
    print(pd.DataFrame(reports).to_string(index=False))
    # Exigir todos os anos
    if len(reports) != len(MUNICIPIOS) * (END_YEAR - START_YEAR + 1):
        raise RuntimeError("Ainda há anos faltando ou vazios — não empacote.")
    return reports


def package_validated_exports():
    import shutil
    staging = LOCAL_DIR / "panels"
    for ibge_code in MUNICIPIOS:
        d = staging / str(ibge_code)
        d.mkdir(parents=True, exist_ok=True)
        for year in range(START_YEAR, END_YEAR + 1):
            src = DRIVE_DIR / f"panel_{ibge_code}_{year}.csv"
            df = pd.read_csv(src)
            if len(df) == 0:
                raise ValueError(f"CSV vazio: {src.name}")
            (d / src.name).write_bytes(src.read_bytes())
    z = Path("/content/ForestGuard_panels_v22.zip")
    shutil.make_archive(str(z.with_suffix("")), "zip", staging)
    print("ZIP:", z)
    if IN_COLAB:
        from google.colab import files  # type: ignore
        files.download(str(z))


# validate_downloaded_exports()
# package_validated_exports()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 30.1 MB/s eta 0:00:00
Mounted at /content/drive
MapBiomas OK: classification_1995 … classification_2023

=== Município 4210100 ===


municipalities_2022_simplified.parquet: 100%|██████████| 20.6M/20.6M [00:00<00:00, 169MB/s]


Grade: 1400 células → pasta Drive: ForestGuard_Panels_v22
  [diagnóstico] 4210100/2000: 1400 features
  Export iniciado: FG_v22_4210100_2000
  [diagnóstico] 4210100/2001: 1400 features
  Export iniciado: FG_v22_4210100_2001
  [diagnóstico] 4210100/2002: 1400 features
  Export iniciado: FG_v22_4210100_2002
  [diagnóstico] 4210100/2003: 1400 features
  Export iniciado: FG_v22_4210100_2003
  [diagnóstico] 4210100/2004: 1400 features
  Export iniciado: FG_v22_4210100_2004
  [diagnóstico] 4210100/2005: 1400 features
  Export iniciado: FG_v22_4210100_2005
  [diagnóstico] 4210100/2006: 1400 features
  Export iniciado: FG_v22_4210100_2006
  [diagnóstico] 4210100/2007: 1400 features
  Export iniciado: FG_v22_4210100_2007
  [diagnóstico] 4210100/2008: 1400 features
  Export iniciado: FG_v22_4210100_2008
  [diagnóstico] 4210100/2009: 1400 features
  Export iniciado: FG_v22_4210100_2009
  [diagnóstico] 4210100/2010: 1400 features
  Export iniciado: FG_v22_4210100_2010
  [diagnóstico] 4210100/2011: